In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install nltk rouge-score bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 105.1 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl si

In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [ ]:
%cd /content/drive/MyDrive/연세_졸업논문/

/content/drive/MyDrive/연세_졸업논문


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/연세_졸업논문/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from bllossom_pad_100_dataset import CustomDataset
from bllossom_alpha_O_train_valid import train, valid, calculate_rouge
from bllossom_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt
from bllossom_run import run

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    'model_name': 'Bllossom/llama-3.2-Korean-Bllossom-3B', # right
    'path': '/content/drive/MyDrive/연세_졸업논문/results/',
    'model_path': '6000_0.5_pad_O_llama-3.2-Korean-Bllossom-3B_best_tot_ver_{}.pt',
    'checkpoint_path': '6000_0.5_pad_O_llama-3.2-Korean-Bllossom-3B_tot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_0.5_pad_O_llama-3.2-Korean-Bllossom-3B_tot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_0.5_pad_O_llama-3.2-Korean-Bllossom-3B_df_test_pred_tot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'tot', # student, tot, cot, basic 확인!!
    'alpha' : 0.5,
    'epochs' :  7,
    'start_epoch': 6, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 2, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 5.667505561510722, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['ver_start'] == 0 and meta_data['start_epoch'] == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['json_output_file'], 'r') as f:
        all_results = json.load(f)

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df = pd.read_csv('/content/drive/MyDrive/연세_졸업논문/{}_student_train_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df = pd.read_csv('/content/drive/MyDrive/연세_졸업논문/ab_train_data_20000_merged.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    if meta_data['start_epoch'] == 0:
        meta_data['save_val_loss'] = 100

    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver
    df = df.sample(frac = 1, random_state = ver).reset_index(drop = True)
    split_idx = int(meta_data['data_num'] * 0.9)
    df_train = df.iloc[:split_idx]
    df_valid = df.iloc[split_idx:int(meta_data['data_num'])]
    # df_train = df.iloc[:5]
    # df_valid = df.iloc[5:10]

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {
            'meta_data' : meta_data,
            'time': None,
            'results': []
        }
    start = time.time()
    all_results = run(ver, meta_data, df_train, df_valid, calculate_rouge, content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, all_results)
    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))

    with open(meta_data['path'] + meta_data['json_output_file'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

    meta_data['start_epoch'] = 0

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


========== version 2 ==========


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/180 [00:00<?, ?B/s]



여기

Epoch: 7
--------------------


  0%|          | 0/5400 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[C_loss : 5.6666]: 100%|██████████| 600/600 [16:42<00:00,  1.67s/it]


best model was saved


In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [ ]:
# l4
import sys
sys.path.append('/content/drive/MyDrive/연세_졸업논문/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from bllossom_pad_100_dataset import CustomDataset
from bllossom_alpha_O_train_valid import train, valid, calculate_rouge
from bllossom_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, test_prompt
from bllossom_run import run
from bllossom_test import pred_test

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    'model_name': 'Bllossom/llama-3.2-Korean-Bllossom-3B', # right
    'path': '/content/drive/MyDrive/연세_졸업논문/results/',
    'model_path': '6000_0.5_pad_O_llama-3.2-Korean-Bllossom-3B_best_tot_ver_{}.pt',
    'checkpoint_path': '6000_0.5_pad_O_llama-3.2-Korean-Bllossom-3B_tot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_0.5_pad_O_llama-3.2-Korean-Bllossom-3B_tot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_0.5_pad_O_llama-3.2-Korean-Bllossom-3B_df_test_pred_tot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'tot', # student, tot, cot, basic 확인!!
    'alpha' : 0.5,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df_test = pd.read_csv('/content/drive/MyDrive/연세_졸업논문/{}_student_test_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df_test = pd.read_csv('/content/drive/MyDrive/연세_졸업논문/ab_test_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver

    # test_df = pred_test(ver, meta_data, df_test, calculate_rouge, tot_prompt1, tot_prompt2, tot_prompt3, tot_prompt4, cot_prompt1, cot_prompt2, basic_prompt)
    test_df = pred_test(ver, meta_data, df_test, content2, test_prompt)

test_df
print(test_df['prediction'][0])

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# alpha만 없는 거

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install nltk rouge-score bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 109.1 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl siz

In [3]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4
%cd /content/drive/MyDrive/연세_졸업논문/

/content/drive/MyDrive/연세_졸업논문


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/연세_졸업논문/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from bllossom_pad_100_dataset import CustomDataset
from bllossom_alpha_X_train_valid import train, valid, calculate_rouge
from bllossom_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt
from bllossom_None_run import run

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    'model_name': 'Bllossom/llama-3.2-Korean-Bllossom-3B', # right
    'path': '/content/drive/MyDrive/연세_졸업논문/results/',
    'model_path': '6000_None_pad_O_llama-3.2-Korean-Bllossom-3B_best_tot_ver_{}.pt',
    'checkpoint_path': '6000_None_pad_O_llama-3.2-Korean-Bllossom-3B_tot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_None_pad_O_llama-3.2-Korean-Bllossom-3B_tot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_None_pad_O_llama-3.2-Korean-Bllossom-3B_df_test_pred_tot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'tot', # student, tot, cot, basic 확인!!
    'alpha' : None,
    'epochs' :  7,
    'start_epoch': 6, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 2, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 11.349520214398702, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['ver_start'] == 0 and meta_data['start_epoch'] == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['json_output_file'], 'r') as f:
        all_results = json.load(f)

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df = pd.read_csv('/content/drive/MyDrive/연세_졸업논문/{}_student_train_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df = pd.read_csv('/content/drive/MyDrive/연세_졸업논문/ab_train_data_20000_merged.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    if meta_data['start_epoch'] == 0:
        meta_data['save_val_loss'] = 100

    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver
    df = df.sample(frac = 1, random_state = ver).reset_index(drop = True)
    split_idx = int(meta_data['data_num'] * 0.9)
    df_train = df.iloc[:split_idx]
    df_valid = df.iloc[split_idx:int(meta_data['data_num'])]
    # df_train = df.iloc[:10]
    # df_valid = df.iloc[10:20]

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {
            'meta_data' : meta_data,
            'time': None,
            'results': []
        }
    start = time.time()
    all_results = run(ver, meta_data, df_train, df_valid, calculate_rouge, content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, all_results)
    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))

    with open(meta_data['path'] + meta_data['json_output_file'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

    meta_data['start_epoch'] = 0

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


========== version 2 ==========


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/180 [00:00<?, ?B/s]



여기

Epoch: 7
--------------------


  0%|          | 0/5400 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[C_loss : 11.347]: 100%|██████████| 600/600 [16:44<00:00,  1.67s/it]


best model was saved


In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4
%cd /content/drive/MyDrive/연세_졸업논문/

/content/drive/MyDrive/연세_졸업논문


In [4]:
# l4
import sys
sys.path.append('/content/drive/MyDrive/연세_졸업논문/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from bllossom_pad_100_dataset import CustomDataset
from bllossom_alpha_X_train_valid import train, valid, calculate_rouge
from bllossom_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, test_prompt
from bllossom_None_run import run
from bllossom_test import pred_test

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    'model_name': 'Bllossom/llama-3.2-Korean-Bllossom-3B', # right
    'path': '/content/drive/MyDrive/연세_졸업논문/results/',
    'model_path': '6000_None_pad_O_llama-3.2-Korean-Bllossom-3B_best_tot_ver_{}.pt',
    'checkpoint_path': '6000_None_pad_O_llama-3.2-Korean-Bllossom-3B_tot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_None_pad_O_llama-3.2-Korean-Bllossom-3B_tot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_None_pad_O_llama-3.2-Korean-Bllossom-3B_df_test_pred_tot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'tot', # student, tot, cot, basic 확인!!
    'alpha' : None,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 1, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df_test = pd.read_csv('/content/drive/MyDrive/연세_졸업논문/{}_student_test_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df_test = pd.read_csv('/content/drive/MyDrive/연세_졸업논문/ab_test_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver

    # test_df = pred_test(ver, meta_data, df_test, calculate_rouge, tot_prompt1, tot_prompt2, tot_prompt3, tot_prompt4, cot_prompt1, cot_prompt2, basic_prompt)
    test_df = pred_test(ver, meta_data, df_test, content2, test_prompt)

test_df
print(test_df['prediction'][0])

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


========== version 1 ==========


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

...

100%|██████████| 1918/1918 [3:49:28<00:00,  7.18s/it]


========== version 2 ==========


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 1918/1918 [4:24:40<00:00,  8.28s/it]


system
당신은 화자의 발언에 공감하는 청자 역할입니다.user
화자의 발화에 대한 공감 답변을 생성하세요.
### 화자의 발화: 엄마, 이번에 주전 선수의 대리로 경기에 출전하게 됐어요! 항상 보결로 벤치에 앉아 모두를 지켜보기만 했었는걸요. 직접 뛸 수 있게 돼서 너무나 기뻐요!
### 청자의 공감 답변:assistant
정말 열심히 노력한 결과라 정말 자랑스럽고 영감받아요! 다음까지도 잘 해내길 바랄께요. 그동안 계속 cheering했는데 이제 진짜 소중한 순간이 되네요. 네가 이렇게 성장하니 정말 tự부분하고 행복해요! 혹시 어떤 팀이라면 얼마든지 이끌어갈 거예요? 너의 꿈과 함께 하는 것이 꼭 나에게 큰 의미야! 고마워요! 내 마음에도 많이 좋았던 순간이었어요. 앞으로 더 많은 successes를 쌓기 위해 힘들 때마다 응원해줄게! 우리 팀에서 승리하기 위해서는 함께 가꾸며 도와주실 거야!
